# 文本变体消融实验 — 纵向对比

控制变量：**模型架构固定为 full**（ID + Image + Text + 跨域），仅切换文本特征。

| 变体 | 文本输入 | 核心问题 |
|------|------|------|
| A: desc | title + description[:1500] | 原始噪声会掩盖信号吗？ |
| B: llm | title + llm_v2 (200词) | LLM 蒸馏是否足够？ |
| C: llmdesc | title + llm_v2 + desc[:400] | 互补是否有增量？ |

关键对比：
- A vs B → LLM 增强的价值
- B vs C → 原文的边际贡献
- A vs C → 总提升幅度

### 1. 路径 / 配置超参数

三个特征文件由同一 BGE-large-en-v1.5 编码器提取，维度均为 1024d，仅在**输入文本内容**上有差异。

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, math, os, json, datetime
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

torch.backends.cudnn.benchmark = True

from cdsr_model import CDSRModel

DATA_DIR = "final"
ID_FEAT_PATH   = os.path.join(DATA_DIR, "features", "item_id_lightgcn_512.npy")
IMG_FEAT_PATH  = os.path.join(DATA_DIR, "features", "image_features_768.npy")
TRAIN_CSV      = os.path.join(DATA_DIR, "train.csv")
VAL_CSV        = os.path.join(DATA_DIR, "val.csv")
TEST_CSV       = os.path.join(DATA_DIR, "test.csv")
ITEM2ID_PATH   = os.path.join(DATA_DIR, "item2id.json")
META_CSV       = os.path.join(DATA_DIR, "item_meta_merged.csv")

SAVE_DIR = os.path.join(DATA_DIR, "checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)

TEXT_VARIANTS = [
    ("desc",    "text_bge_desc_1024.npy",    "A: 纯原文"),
    ("llm",     "text_bge_llm_1024.npy",     "B: 纯LLM"),
    ("llmdesc", "text_bge_llmdesc_1024.npy", "C: LLM+原文"),
]

RESULT_PATH = os.path.join(SAVE_DIR, "ablation_text_results.txt")

print("文本变体消融 — 三个变体:")
for tag, fname, desc in TEXT_VARIANTS:
    print(f"  {tag:8s} → {fname}  ({desc})")

文本变体消融 — 三个变体:
  desc     → text_bge_desc_1024.npy  (A: 纯原文)
  llm      → text_bge_llm_1024.npy  (B: 纯LLM)
  llmdesc  → text_bge_llmdesc_1024.npy  (C: LLM+原文)


文本变体消融使用与架构消融**完全相同的超参数**，且三个变体之间不调整任何参数，确保对比的公平性：

In [2]:
D_MODEL         = 256
N_HEADS         = 4
N_LAYERS        = 2
DROPOUT         = 0.2
MAX_SEQ_LEN     = 200
BATCH_SIZE      = 256
LR              = 1.4e-3
TEXT_ABL_EPOCHS = 3
PATIENCE        = 3
LAMBDA1         = 0.3
LAMBDA2         = 0.1
LABEL_SMOOTH    = 0.0
USE_AMP         = True
NUM_WORKERS     = 8
EVAL_BS         = 256       # 评估用小 BS
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}  AMP={USE_AMP}  BS={BATCH_SIZE}  EvalBS={EVAL_BS}  Epochs per variant: {TEXT_ABL_EPOCHS}")

Device: cuda  AMP=True  BS=256  EvalBS=256  Epochs per variant: 3


### 2. 固定 / 变体特征

- **ID Embedding** (`item_id_lightgcn_512.npy`): LightGCN 3 层图卷积 + BPR Loss，43,528 个物品的 512d 协同向量。
- **Image Embedding** (`image_features_768.npy`): CLIP ViT-L/14 提取，768d 视觉特征

- **Text Embedding**: 三种变体

In [3]:
print("加载 ID + Image 特征...")
id_feats = torch.from_numpy(np.load(ID_FEAT_PATH)).float()
img_feats = torch.from_numpy(np.load(IMG_FEAT_PATH)).float()
print(f"ID:    {id_feats.shape}")
print(f"Image: {img_feats.shape}")

with open(ITEM2ID_PATH) as f:
    item2id = json.load(f)
N_ITEMS = len(item2id)
print(f"物品总数: {N_ITEMS}")

meta_df = pd.read_csv(META_CSV)
movie_mask = torch.zeros(N_ITEMS, dtype=torch.bool)
book_mask  = torch.zeros(N_ITEMS, dtype=torch.bool)
movie_ids = meta_df[meta_df['domain'] == 0].index.tolist()
book_ids  = meta_df[meta_df['domain'] == 1].index.tolist()
movie_mask[movie_ids] = True
book_mask[book_ids]  = True
print(f"电影: {movie_mask.sum().item()}  图书: {book_mask.sum().item()}")

加载 ID + Image 特征...
ID:    torch.Size([43528, 512])
Image: torch.Size([43528, 768])
物品总数: 43528
电影: 21280  图书: 22248


### 3.样本生成
与架构消融完全相同的 `build_samples` / `build_eval_samples` 实现。


In [4]:
def build_user_sequences(df):
    user_seqs = {}
    for uid, g in df.groupby("user_id"):
        g = g.sort_values("timestamp")
        uid = int(uid)
        sx, sy, sxy = [], [], []
        for _, row in g.iterrows():
            ts = row["timestamp"]
            iid = int(row["item_id"])
            dom = int(row["domain"])
            if dom == 0:
                sx.append((ts, iid))
            else:
                sy.append((ts, iid))
            sxy.append((ts, iid, dom))
        user_seqs[uid] = {"sx": sx, "sy": sy, "sxy": sxy}
    return user_seqs

def build_samples(user_seqs):
    samples = []
    for uid, seqs in user_seqs.items():
        sx, sy, sxy = seqs["sx"], seqs["sy"], seqs["sxy"]
        if len(sxy) < 2:
            continue
        sx_ptr = sy_ptr = 0
        for k in range(1, len(sxy)):
            ts_k = sxy[k][0]
            while sx_ptr < len(sx) and sx[sx_ptr][0] < ts_k:
                sx_ptr += 1
            while sy_ptr < len(sy) and sy[sy_ptr][0] < ts_k:
                sy_ptr += 1
            samples.append(dict(sx_ids=[x[1] for x in sx[:sx_ptr]],
                                sy_ids=[y[1] for y in sy[:sy_ptr]],
                                sxy_ids=[z[1] for z in sxy[:k]],
                                target=sxy[k][1], target_domain=sxy[k][2], seq_type=2))
        sx_to_sxy, sx_cnt = {}, 0
        for p, (ts, iid, dom) in enumerate(sxy):
            if dom == 0:
                sx_to_sxy[sx_cnt] = p; sx_cnt += 1
        sy_ptr = 0
        for i in range(1, len(sx)):
            while sy_ptr < len(sy) and sy[sy_ptr][0] < sx[i][0]:
                sy_ptr += 1
            samples.append(dict(sx_ids=[x[1] for x in sx[:i]],
                                sy_ids=[y[1] for y in sy[:sy_ptr]],
                                sxy_ids=[z[1] for z in sxy[:sx_to_sxy[i]]],
                                target=sx[i][1], target_domain=0, seq_type=0))
        sy_to_sxy, sy_cnt = {}, 0
        for p, (ts, iid, dom) in enumerate(sxy):
            if dom == 1:
                sy_to_sxy[sy_cnt] = p; sy_cnt += 1
        sx_ptr = 0
        for j in range(1, len(sy)):
            while sx_ptr < len(sx) and sx[sx_ptr][0] < sy[j][0]:
                sx_ptr += 1
            samples.append(dict(sx_ids=[x[1] for x in sx[:sx_ptr]],
                                sy_ids=[y[1] for y in sy[:j]],
                                sxy_ids=[z[1] for z in sxy[:sy_to_sxy[j]]],
                                target=sy[j][1], target_domain=1, seq_type=1))
    return samples

def build_eval_samples(context_seqs, target_seqs):
    samples = []
    for uid, t_seqs in target_seqs.items():
        if uid not in context_seqs:
            continue
        ctx = context_seqs[uid]
        sx = ctx["sx"] + t_seqs["sx"]
        sy = ctx["sy"] + t_seqs["sy"]
        sxy = ctx["sxy"] + t_seqs["sxy"]
        if len(sxy) < 2:
            continue
        k = len(sxy) - 1
        ts_k = sxy[k][0]
        sx_ptr = len([x for x in sx if x[0] < ts_k])
        sy_ptr = len([y for y in sy if y[0] < ts_k])
        samples.append(dict(sx_ids=[x[1] for x in sx[:sx_ptr]],
                            sy_ids=[y[1] for y in sy[:sy_ptr]],
                            sxy_ids=[z[1] for z in sxy[:k]],
                            target=sxy[k][1], target_domain=sxy[k][2]))
    return samples

print("构建样本...")
train_seqs = build_user_sequences(pd.read_csv(TRAIN_CSV))
val_seqs   = build_user_sequences(pd.read_csv(VAL_CSV))
test_seqs  = build_user_sequences(pd.read_csv(TEST_CSV))

train_samples = build_samples(train_seqs)
val_samples = build_eval_samples(train_seqs, val_seqs)

train_val_seqs = {}
for uid in set(train_seqs.keys()) | set(val_seqs.keys()):
    ts = train_seqs.get(uid, {"sx": [], "sy": [], "sxy": []})
    vs = val_seqs.get(uid, {"sx": [], "sy": [], "sxy": []})
    train_val_seqs[uid] = {"sx": ts["sx"] + vs["sx"], "sy": ts["sy"] + vs["sy"], "sxy": ts["sxy"] + vs["sxy"]}
test_samples = build_eval_samples(train_val_seqs, test_seqs)

print(f"Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}")

构建样本...
Train: 1235810  Val: 20030  Test: 20030


In [5]:
class CDSRDataset(Dataset):
    def __init__(self, samples, max_len=MAX_SEQ_LEN, is_train=True):
        self.samples = samples; self.max_len = max_len; self.is_train = is_train
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        def pad(ids):
            ids = ids[-self.max_len:]
            n = len(ids)
            return torch.tensor(ids + [0] * (self.max_len - n), dtype=torch.long), \
                   torch.tensor([1] * n + [0] * (self.max_len - n), dtype=torch.long)
        sx, sx_mask   = pad(s["sx_ids"])
        sy, sy_mask   = pad(s["sy_ids"])
        sxy, sxy_mask = pad(s["sxy_ids"])
        if self.is_train:
            return (sx, sx_mask, sy, sy_mask, sxy, sxy_mask,
                    torch.tensor(s["target"], dtype=torch.long),
                    torch.tensor(s["target_domain"], dtype=torch.long),
                    torch.tensor(s["seq_type"], dtype=torch.long))
        else:
            return (sx, sx_mask, sy, sy_mask, sxy, sxy_mask,
                    torch.tensor(s["target"], dtype=torch.long),
                    torch.tensor(s["target_domain"], dtype=torch.long))

def collate_train(batch):
    sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom, seq_type = zip(*batch)
    return (torch.stack(list(sx)), torch.stack(list(sx_mask)),
            torch.stack(list(sy)), torch.stack(list(sy_mask)),
            torch.stack(list(sxy)), torch.stack(list(sxy_mask)),
            torch.stack(list(target)), torch.stack(list(tgt_dom)),
            torch.stack(list(seq_type)))

def collate_eval(batch):
    sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom = zip(*batch)
    return (torch.stack(list(sx)), torch.stack(list(sx_mask)),
            torch.stack(list(sy)), torch.stack(list(sy_mask)),
            torch.stack(list(sxy)), torch.stack(list(sxy_mask)),
            torch.stack(list(target)), torch.stack(list(tgt_dom)))

train_ds = CDSRDataset(train_samples, is_train=True)
val_ds   = CDSRDataset(val_samples,   is_train=False)
test_ds  = CDSRDataset(test_samples,  is_train=False)

NUM_WORKERS_ACTUAL = min(NUM_WORKERS, os.cpu_count() or 4)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_train, num_workers=NUM_WORKERS_ACTUAL,
                           pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=EVAL_BS, shuffle=False,
                           collate_fn=collate_eval, num_workers=NUM_WORKERS_ACTUAL, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=EVAL_BS, shuffle=False,
                           collate_fn=collate_eval, num_workers=NUM_WORKERS_ACTUAL, pin_memory=True)

print(f"Train batches/epoch: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}")

Train batches/epoch: 4828  Val: 79  Test: 79


### 5. 训练 & 评估函数

In [6]:
def train_one_text_variant(tag, feat_fname, desc):
    """用指定文本特征训练 full 模型，返回评估指标。"""
    print(f"\n{'='*60}")
    print(f"  文本变体: {desc}  ({feat_fname})")
    print(f"{'='*60}")

    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    tex_feats = torch.from_numpy(
        np.load(os.path.join(DATA_DIR, "features", feat_fname))
    ).float()
    print(f"Text features: {tex_feats.shape}")

    model = CDSRModel(N_ITEMS, id_feats, img_feats, tex_feats,
                       movie_mask, book_mask,
                       d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                       dropout=DROPOUT, max_len=MAX_SEQ_LEN,
                       ablation="full").to(DEVICE)

    n_p = sum(p.numel() for p in model.parameters())
    print(f"参数量: {n_p/1e6:.1f}M")

    scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01,
                                   fused=True if DEVICE == "cuda" else False)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

    @torch.no_grad()
    def evaluate(loader, k=10):
        model.eval()
        hits, ndcg_sum, mrr_sum, total = 0, 0, 0, 0
        dh, dt = {0: 0, 1: 0}, {0: 0, 1: 0}
        for sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom in loader:
            sx, sx_mask = sx.to(DEVICE), sx_mask.to(DEVICE)
            sy, sy_mask = sy.to(DEVICE), sy_mask.to(DEVICE)
            sxy, sxy_mask = sxy.to(DEVICE), sxy_mask.to(DEVICE)
            target = target.to(DEVICE); tgt_dom = tgt_dom.to(DEVICE)
            out = model(sx, sx_mask, sy, sy_mask, sxy, sxy_mask, return_cross=True)
            for i in range(len(target)):
                dom = tgt_dom[i].item()
                if dom == 0:
                    score = out["P_X"][i] + LAMBDA1 * out["P_Y_to_X"][i] + LAMBDA2 * out["P_XY_to_X"][i]
                else:
                    score = out["P_X_to_Y"][i] + LAMBDA1 * out["P_Y"][i] + LAMBDA2 * out["P_XY_to_Y"][i]
                _, topk = torch.topk(score, k)
                topk = topk.cpu().tolist()
                tgt = target[i].item()
                if tgt in topk:
                    rank = topk.index(tgt) + 1
                    hits += 1; dh[dom] += 1
                    ndcg_sum += 1.0 / math.log2(rank + 1)
                    mrr_sum += 1.0 / rank
                total += 1; dt[dom] += 1
        hr = hits / total if total else 0
        ndcg = ndcg_sum / total if total else 0
        mrr = mrr_sum / total if total else 0
        return hr, ndcg, mrr, total, dh, dt

    # 断点续训
    best_path = os.path.join(SAVE_DIR, f"text_abl_{tag}_best.pt")
    ckpt_path = os.path.join(SAVE_DIR, f"text_abl_{tag}_ckpt.pt")
    start_epoch = 1
    best_val_hr = 0.0
    best_val_mrr = 0.0
    no_improve = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if scaler is not None and ckpt.get("scaler"):
            scaler.load_state_dict(ckpt["scaler"])
        start_epoch = ckpt["epoch"] + 1
        best_val_hr = ckpt.get("best_val_hr", 0.0)
        best_val_mrr = ckpt.get("best_val_mrr", 0.0)
        no_improve = ckpt["no_improve"]
        print(f"  从断点恢复: epoch={start_epoch}, best_val_hr={best_val_hr:.4f}")
    else:
        print("  从头开始训练")

    print(f"  {'Epoch':>6} {'TrainLoss':>10} {'ValHR@10':>9} {'ValNDCG':>8} {'ValMRR':>8} "
          f"{'TestHR@10':>10} {'TestNDCG':>9} {'TestMRR':>9} {'MovieHR':>8} {'BookHR':>8}")
    print("  " + "-" * 95)

    for epoch in range(start_epoch, TEXT_ABL_EPOCHS + 1):
        model.train()
        total_loss, n_batch = 0, 0
        pbar = tqdm(train_loader, desc=f"  [{tag}] Epoch {epoch}/{TEXT_ABL_EPOCHS}")

        for sx, sx_mask, sy, sy_mask, sxy, sxy_mask, target, tgt_dom, seq_type in pbar:
            sx, sx_mask = sx.to(DEVICE), sx_mask.to(DEVICE)
            sy, sy_mask = sy.to(DEVICE), sy_mask.to(DEVICE)
            sxy, sxy_mask = sxy.to(DEVICE), sxy_mask.to(DEVICE)
            target, seq_type = target.to(DEVICE), seq_type.to(DEVICE)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    out = model(sx, sx_mask, sy, sy_mask, sxy, sxy_mask, return_cross=False)
                loss = torch.tensor(0.0, device=DEVICE)
                for st, wt, key in [(0, 1.0, "P_X"), (1, LAMBDA1, "P_Y"), (2, LAMBDA2, "P_XY")]:
                    m = (seq_type == st)
                    if m.any():
                        loss = loss + wt * criterion(out[key][m], target[m])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(sx, sx_mask, sy, sy_mask, sxy, sxy_mask, return_cross=False)
                loss = torch.tensor(0.0, device=DEVICE)
                for st, wt, key in [(0, 1.0, "P_X"), (1, LAMBDA1, "P_Y"), (2, LAMBDA2, "P_XY")]:
                    m = (seq_type == st)
                    if m.any():
                        loss = loss + wt * criterion(out[key][m], target[m])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item(); n_batch += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / n_batch

        val_hr, val_ndcg, val_mrr, _, val_dh, val_dt = evaluate(val_loader)
        test_hr, test_ndcg, test_mrr, _, test_dh, test_dt = evaluate(test_loader)
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        movie_hr = test_dh[0] / test_dt[0] if test_dt[0] else 0
        book_hr = test_dh[1] / test_dt[1] if test_dt[1] else 0

        print(f"  {epoch:>6} {avg_loss:>10.4f} {val_hr:>9.4f} {val_ndcg:>8.4f} {val_mrr:>8.4f} "
              f"{test_hr:>10.4f} {test_ndcg:>9.4f} {test_mrr:>9.4f} {movie_hr:>8.4f} {book_hr:>8.4f}")
        scheduler.step(val_hr)

        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                       scheduler=scheduler.state_dict(),
                       scaler=scaler.state_dict() if scaler else {},
                       epoch=epoch, best_val_hr=best_val_hr, best_val_mrr=best_val_mrr,
                       no_improve=no_improve), ckpt_path)

        if val_hr > best_val_hr + 1e-5:
            best_val_hr = val_hr; no_improve = 0
            torch.save(model.state_dict(), best_path)
            print("  -> saved (best hr)")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  Early stop at epoch {epoch}"); break

        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"text_abl_{tag}_best_val.pt"))

    # 最终评估
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()
    eval_val_hr, eval_val_ndcg, eval_val_mrr, _, _, _ = evaluate(val_loader)
    eval_test_hr, eval_test_ndcg, eval_test_mrr, _, test_dh, test_dt = evaluate(test_loader)

    print(f"  Final: Val HR={eval_val_hr:.4f} NDCG={eval_val_ndcg:.4f} MRR={eval_val_mrr:.4f}  "
          f"Test HR={eval_test_hr:.4f} NDCG={eval_test_ndcg:.4f} MRR={eval_test_mrr:.4f}")

    del tex_feats, model, optimizer, scheduler
    if scaler is not None:
        del scaler
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return dict(tag=tag, desc=desc,
                val_hr=eval_val_hr, val_ndcg=eval_val_ndcg, val_mrr=eval_val_mrr,
                test_hr=eval_test_hr, test_ndcg=eval_test_ndcg, test_mrr=eval_test_mrr,
                test_movie_hr=test_dh[0]/test_dt[0] if test_dt[0] else 0,
                test_book_hr=test_dh[1]/test_dt[1] if test_dt[1] else 0)

### 6. 依次训练 3 个文本变体

In [ ]:
print(f"\n文本变体消融开始: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

results = []
for tag, fname, desc in TEXT_VARIANTS:
    r = train_one_text_variant(tag, fname, desc)
    results.append(r)
    print(f"\n--- 已完成 {len(results)}/{len(TEXT_VARIANTS)} ---")
    for r_ in results:
        print(f"  {r_['desc']:14s}: Val HR={r_['val_hr']:.4f}  Test HR={r_['test_hr']:.4f}  MRR={r_['test_mrr']:.4f}")

print(f"\n文本变体消融完成: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
print("\n" + "=" * 95)
print("  文本变体消融实验结果")
print("=" * 95)
print(f"{'Variant':<20} {'ValHR':>7} {'ValNDCG':>8} {'ValMRR':>8} "
      f"{'TestHR':>8} {'TestNDCG':>9} {'TestMRR':>9} {'MovieHR':>8} {'BookHR':>8}")
print("-" * 95)

for r in results:
    print(f"{r['desc']:<20} {r['val_hr']:>7.4f} {r['val_ndcg']:>8.4f} {r['val_mrr']:>8.4f} "
          f"{r['test_hr']:>8.4f} {r['test_ndcg']:>9.4f} {r['test_mrr']:>9.4f} "
          f"{r['test_movie_hr']:>8.4f} {r['test_book_hr']:>8.4f}")

print("-" * 95)

# 核心对比
if len(results) == 3:
    ra, rb, rc = results

    print("\n核心对比:")
    print(f"  A→B (LLM增强价值):     HR@10 {rb['test_hr']-ra['test_hr']:+.4f}  "
          f"NDCG@10 {rb['test_ndcg']-ra['test_ndcg']:+.4f}  MRR {rb['test_mrr']-ra['test_mrr']:+.4f}")
    print(f"  B→C (原文边际贡献):    HR@10 {rc['test_hr']-rb['test_hr']:+.4f}  "
          f"NDCG@10 {rc['test_ndcg']-rb['test_ndcg']:+.4f}  MRR {rc['test_mrr']-rb['test_mrr']:+.4f}")
    print(f"  A→C (总提升):           HR@10 {rc['test_hr']-ra['test_hr']:+.4f}  "
          f"NDCG@10 {rc['test_ndcg']-ra['test_ndcg']:+.4f}  MRR {rc['test_mrr']-ra['test_mrr']:+.4f}")

    best_idx = max(range(3), key=lambda i: results[i]['test_hr'])
    print(f"\n结论: 最优文本变体 = {results[best_idx]['desc']}  (Test HR@10={results[best_idx]['test_hr']:.4f})")

# 保存
with open(RESULT_PATH, "w", encoding="utf-8") as f:
    f.write(f"文本变体消融结果 — {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n")
    for r in results:
        f.write(f"{r['desc']}: ValHR={r['val_hr']:.4f} ValNDCG={r['val_ndcg']:.4f} ValMRR={r['val_mrr']:.4f}  "
                f"TestHR={r['test_hr']:.4f} TestNDCG={r['test_ndcg']:.4f} TestMRR={r['test_mrr']:.4f}  "
                f"MovieHR={r['test_movie_hr']:.4f} BookHR={r['test_book_hr']:.4f}\n")
    if len(results) == 3:
        f.write("\n核心对比:\n")
        f.write(f"  A→B (LLM增强):  HR@10 {rb['test_hr']-ra['test_hr']:+.4f}  MRR {rb['test_mrr']-ra['test_mrr']:+.4f}\n")
        f.write(f"  B→C (原文边际): HR@10 {rc['test_hr']-rb['test_hr']:+.4f}  MRR {rc['test_mrr']-rb['test_mrr']:+.4f}\n")
        f.write(f"  A→C (总提升):   HR@10 {rc['test_hr']-ra['test_hr']:+.4f}  MRR {rc['test_mrr']-ra['test_mrr']:+.4f}\n")

print(f"\n结果已保存至 {RESULT_PATH}")
print("\n文本变体消融实验全部完成。")